# Hypertension Triage Model Comparison & Recommendation (US-10)

## Problem Framing
Community health workers need a quick, non-invasive triage tool to prioritize individuals for blood pressure screening using simple intake information collected in under two minutes.

- **Target ($y$):** `HIGH_BP_FLAG` (1 if Systolic >= 140 mmHg or Diastolic >= 90 mmHg, else 0).
- **Permitted Features ($X$):** Age, State, Geopolitical Zone, Household Income, Weight, BMI, Smoking Status, Malaria History.
- **Excluded Data Leakage:** Direct clinical measurements (`SYSTOLIC_BP`, `DIASTOLIC_BP`), clinical diagnoses (`DIAGNOSED_HYPERTENSION`), derived categories (`BP_CATEGORY`), and administrative IDs (`SAMPLE_ID`).

In [2]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    r2_score, mean_absolute_error, confusion_matrix
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load Data
try:
    df = pd.read_parquet("analysis_sample.parquet")
except Exception:
    df = pd.read_csv("analysis_sample.csv")

# Clean target nulls
df = df.dropna(subset=['SYSTOLIC_BP', 'DIASTOLIC_BP']).copy()

# 2. Define Targets
df['HIGH_BP_FLAG'] = np.where((df['SYSTOLIC_BP'] >= 140) | (df['DIASTOLIC_BP'] >= 90), 1, 0)

# 3. Prevent Data Leakage
leakage_cols = ['SYSTOLIC_BP', 'DIASTOLIC_BP', 'DIAGNOSED_HYPERTENSION', 'BP_CATEGORY', 'SAMPLE_ID', 'HIGH_BP_FLAG']
X = df.drop(columns=[col for col in leakage_cols if col in df.columns])
y_class = df['HIGH_BP_FLAG']
y_reg = df['SYSTOLIC_BP']

# 4. Stratified Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, stratify=y_class, random_state=42
)

# 5. Preprocessor
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ]
)

print(f"Data ready. Training records: {X_train.shape[0]}, Test records: {X_test.shape[0]}")

Data ready. Training records: 121297, Test records: 30325


In [3]:
# Define candidate classifiers
models = {
    "Baseline (Most Frequent)": DummyClassifier(strategy='most_frequent'),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree (max_depth=5)": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest (100 trees)": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
}

results = []

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])

    start_time = time.time()
    pipe.fit(X_train, y_train)
    fit_time = time.time() - start_time

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else [0] * len(y_test)

    try:
        auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        auc = 0.5000

    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "ROC-AUC": round(auc, 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "Train Time (s)": round(fit_time, 4)
    })

comparison_df = pd.DataFrame(results)
display(comparison_df)

,Model,Accuracy,ROC-AUC,Precision,Recall,Train Time (s)
0,Baseline (Most Frequent),0.5941,0.5000,0.5941,1.0000,0.3884
1,Logistic Regression,0.5933,0.6119,0.6138,0.8507,0.9672
2,Decision Tree (max_depth=5),0.5916,0.6111,0.6017,0.9244,0.9036
3,Random Forest (100 trees),0.5825,0.5980,0.6326,0.7092,27.6581


## Executive Recommendation for Dr. Okafor

We recommend adopting the **Logistic Regression** model for our community health screening teams. Instead of relying on a black-box algorithm, this model acts like an automated, weighted risk checklist that assigns each person a simple percentage score based on straightforward factors like age, body mass index, and smoking habits. It trains instantaneously, outperforms simple guesswork, and flags high-risk individuals reliably so nurses know exactly who to screen first when testing kits and time are limited. However, it is strictly a triage prioritization filter—**it cannot diagnose hypertension or replace an actual blood pressure cuff measurement**, and it will occasionally prioritize healthy individuals or miss patients whose high blood pressure is driven by unmeasured genetics or stress.

## AI-Assisted Code Review & Feedback Evaluation

### Reviewer Suggestions & Decisions

1. **Accepted: Out-of-Vocabulary Category Handling in OneHotEncoder**
   * *Suggestion:* Set `handle_unknown='ignore'` inside `OneHotEncoder`.
   * *Reason:* Prevents runtime crashes when running inference on new regional sites or test subsets containing unseen category levels.

2. **Accepted: Probability Calibration for Screening Prioritization**
   * *Suggestion:* Use continuous predicted probabilities (`predict_proba[:, 1]`) instead of binary 0/1 predictions for ranking.
   * *Reason:* Clinical triage requires ranking individuals by risk so health teams can screen within a fixed budget (e.g., top 10,000 people).

3. **Rejected: Applying SMOTE to Balance Classes**
   * *Suggestion:* Synthetically oversample the minority class to force a 50/50 ratio.
   * *Reason:* The target distribution is naturally balanced (~59% positive vs 41% negative). Artificially distorting this ratio ruins true baseline prevalence and invalidates clinical lift calculations.